In [1]:
import numpy as np
import pandas as pd

# Đọc file CSV
file_path = "D:/code/gr1/data/dataset/test_network.csv"
data = pd.read_csv(file_path)

# Hiển thị các cột trong file để kiểm tra
print("Các cột trong dữ liệu:")
print(data.columns)

# Lấy các đặc trưng cần thiết giống với khi huấn luyện
list_features = [
    'bidirectional_duration_ms', 'src2dst_duration_ms',
    'dst2src_duration_ms', 'src2dst_packets', 'dst2src_packets',
    'src2dst_bytes', 'dst2src_bytes', 'src2dst_max_ps', 'src2dst_min_ps',
    'src2dst_mean_ps', 'src2dst_stddev_ps', 'dst2src_max_ps', 'dst2src_min_ps',
    'dst2src_mean_ps', 'dst2src_stddev_ps', 'bidirectional_mean_piat_ms',
    'bidirectional_stddev_piat_ms', 'bidirectional_max_piat_ms', 
    'bidirectional_min_piat_ms','src2dst_mean_piat_ms', 'src2dst_stddev_piat_ms', 
    'src2dst_max_piat_ms', 'src2dst_min_piat_ms', 'dst2src_mean_piat_ms',
    'dst2src_stddev_piat_ms', 'dst2src_max_piat_ms', 'dst2src_min_piat_ms', 
    'bidirectional_fin_packets', 'src2dst_fin_packets', 'dst2src_fin_packets', 
    'bidirectional_syn_packets', 'src2dst_syn_packets', 'dst2src_syn_packets',
    'bidirectional_rst_packets', 'src2dst_rst_packets', 'dst2src_rst_packets',
    'bidirectional_psh_packets', 'src2dst_psh_packets', 'dst2src_psh_packets', 
    'bidirectional_ack_packets', 'src2dst_ack_packets', 'dst2src_ack_packets', 
    'bidirectional_urg_packets', 'src2dst_urg_packets', 
    'bidirectional_cwr_packets', 'src2dst_cwr_packets', 
    'bidirectional_ece_packets', 'src2dst_ece_packets', 'Stage'
]

# Lọc các đặc trưng cần thiết
df = data[list_features].copy()

Các cột trong dữ liệu:
Index(['id', 'expiration_id', 'src_ip', 'src_mac', 'src_oui', 'src_port',
       'dst_ip', 'dst_mac', 'dst_oui', 'dst_port', 'protocol', 'ip_version',
       'vlan_id', 'tunnel_id', 'bidirectional_first_seen_ms',
       'bidirectional_last_seen_ms', 'bidirectional_duration_ms',
       'bidirectional_packets', 'bidirectional_bytes', 'src2dst_first_seen_ms',
       'src2dst_last_seen_ms', 'src2dst_duration_ms', 'src2dst_packets',
       'src2dst_bytes', 'dst2src_first_seen_ms', 'dst2src_last_seen_ms',
       'dst2src_duration_ms', 'dst2src_packets', 'dst2src_bytes',
       'bidirectional_min_ps', 'bidirectional_mean_ps',
       'bidirectional_stddev_ps', 'bidirectional_max_ps', 'src2dst_min_ps',
       'src2dst_mean_ps', 'src2dst_stddev_ps', 'src2dst_max_ps',
       'dst2src_min_ps', 'dst2src_mean_ps', 'dst2src_stddev_ps',
       'dst2src_max_ps', 'bidirectional_min_piat_ms',
       'bidirectional_mean_piat_ms', 'bidirectional_stddev_piat_ms',
       'bidirectional

In [9]:
# Define the stage mapping
def get_stage_mapping():
    return {
        'Benign': 0,        
        'Reconnaissance': 1,     
        'Establish Foothold': 2,
        'Lateral Movement': 3,
        'Data Exfiltration': 4,
        'Cover up': 5
    }

In [10]:
# Map stages to labels
def map_stages_to_labels(day_data):
    stage_mapping = get_stage_mapping()
    day_data['Label'] = day_data['Stage'].map(stage_mapping)
    return day_data

In [11]:
# Define the score matrix
def get_score_matrix():
    return {
        'Attack': {
            0: [(2.0, 2.5), (2.5, 3.0), (2.5, 3.0), (2.5, 3.0), (2.5, 3.0)],
            1: [(1.0, 1.5), (1.0, 1.5), (2.0, 2.5), (2.0, 2.5), (2.5, 3.5)],
            2: [(1.5, 2.0), (2.0, 2.5), (2.5, 3.5), (3.5, 5.0), (3.0, 5.0)],
            3: [(1.5, 2.5), (2.0, 3.0), (2.5, 3.5), (3.5, 4.5), (3.5, 4.5)],
            4: [(1.5, 2.5), (1.5, 2.5), (3.5, 4.5), (3.5, 4.5), (3.5, 4.5)]
        },
        # 'Defender': {
        #     0: [(8.5, 9.5), (8.5, 9.5), (7.5, 8.5), (7.5, 8.5), (7.0, 8.0)],
        #     1: [(7.5, 8.5), (7.5, 8.5), (7.5, 8.5), (7.5, 8.5), (7.5, 8.5)],
        #     2: [(7.5, 8.5), (7.5, 8.5), (7.5, 8.5), (7.5, 8.5), (7.5, 8.5)],
        #     3: [(6.5, 7.5), (6.5, 7.5), (6.5, 7.5), (6.5, 7.5), (6.5, 7.5)],
        #     4: [(7.5, 8.5), (7.5, 8.5), (7.5, 8.5), (7.5, 8.5), (7.5, 8.5)]
        # }
    }

In [12]:
# Generate random points based on the matrix
def generate_random_points(label, role, matrix, num_points=1):
    if label not in matrix[role]:
        raise ValueError(f"Label '{label}' does not exist in the matrix for role '{role}'.")

    ranges = matrix[role][label]
    random_points = []
    for _ in range(num_points):
        point = [round(np.random.uniform(low, high), 2) for low, high in ranges]
        random_points.append(point)

    return random_points

In [13]:
# Calculate scores for each row of data
def calculate_scores_for_row(row, matrix):
    label = row['Label']
    attack_points = generate_random_points(label, 'Attack', matrix)
    defender_points = generate_random_points(label, 'Defender', matrix)
    results = []
    for attack, defender in zip(attack_points, defender_points):
        result_row = {
            "Label": label,
            **{f"Attack_Feature_{i+1}": value for i, value in enumerate(attack)},
            **{f"Defender_Feature_{i+1}": value for i, value in enumerate(defender)}
        }
        results.append(result_row)
    return results


In [14]:
# Calculate scores for each day
def calculate_scores_for_day(day_data):
    day_data = map_stages_to_labels(day_data)
    matrix = get_score_matrix()
    results = []
    for idx, row in day_data.iterrows():
        results.extend(calculate_scores_for_row(row, matrix))
    return pd.DataFrame(results)

In [16]:
# Chia dữ liệu thành 7 ngày mà không dựa vào timestamp
df['day'] = df.index % 7  # Chia đều theo thứ tự dòng

In [17]:
# Tính điểm cho từng ngày và lưu kết quả
all_results = []
for day in range(7):
    day_data = df[df['day'] == day]
    day_results = calculate_scores_for_day(day_data)
    day_results['day'] = day
    all_results.append(day_results)

# Kết hợp kết quả từ tất cả các ngày
final_results = pd.concat(all_results, ignore_index=True)

# Hiển thị kết quả
print(final_results)

    Label  Attack_Feature_1  Attack_Feature_2  Attack_Feature_3  \
0       3              2.14              2.89              3.16   
1       3              2.16              2.93              3.31   
2       0              2.41              2.52              2.89   
3       0              2.38              2.70              2.57   
4       1              1.13              1.36              2.30   
5       2              1.97              2.46              2.79   
6       3              1.85              2.89              2.85   
7       3              2.05              2.33              3.28   
8       0              2.47              2.75              2.58   
9       1              1.13              1.44              2.05   
10      1              1.38              1.15              2.31   
11      3              2.28              2.21              2.75   
12      3              1.93              2.55              2.57   
13      0              2.45              2.97              2.8

C:\Users\nguye\AppData\Local\Temp\ipykernel_26644\3307007467.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  day_data['Label'] = day_data['Stage'].map(stage_mapping)
C:\Users\nguye\AppData\Local\Temp\ipykernel_26644\3307007467.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  day_data['Label'] = day_data['Stage'].map(stage_mapping)
C:\Users\nguye\AppData\Local\Temp\ipykernel_26644\3307007467.py:4: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .lo

In [18]:
# Save the changes and generated points to a new CSV file
output_file_path = 'D:/code/gr1/data/dataset/Book1.csv'
final_results.to_csv(output_file_path, index=False)